# Customer Purchase Prediction — Scalable MLlib Pipeline
## Behavioral Segmentation and Conversion Modeling with PySpark

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 3–5 — Data Preparation · Modeling · Evaluation |
| **Module** | 9 — Big Data & Spark (Alkemy Bootcamp) |
| **Dataset** | UCI Adult Census Income · 32,561 records · UCI ML Repository |
| **Date** | 2026-03 |

---

> **Executive Summary:**
> This case applies PySpark MLlib to predict whether a worker earns above USD 50K/year
> (Logistic Regression) and to segment socioeconomic profiles (KMeans k=4),
> integrated in an end-to-end MLlib Pipeline. Results provide a replicable framework
> for workforce income segmentation — applicable to HR analytics, public policy,
> and income-based audience targeting.

## Table of Contents

1. [Environment Setup & Data Loading](#1-environment-setup--data-loading)
2. [Data Exploration & Cleaning](#2-data-exploration--cleaning)
3. [Feature Engineering & Preprocessing](#3-feature-engineering--preprocessing)
4. [Supervised Model — Logistic Regression](#4-supervised-model--logistic-regression)
5. [Unsupervised Model — KMeans](#5-unsupervised-model--kmeans)
6. [Results Comparison & Business Synthesis](#6-results-comparison--business-synthesis)
7. [LEAN Filter — Waste Elimination Review](#7-lean-filter--waste-elimination-review)
8. [Decisions Log](#8-decisions-log)
9. [Next Steps](#9-next-steps)
10. [Spark SQL — JOIN Practice](#10-spark-sql--join-practice)
11. [Parameter Optimization — P4 Consigna Compliance](#11-parameter-optimization--p4-consigna-compliance)

---
## 1. Environment Setup & Data Loading
### CRISP-DM Phase 2 — Data Understanding

In [ ]:
# ===== Environment Setup =====
import os
import sys
import warnings

# Force Spark workers to use the same Python as this notebook
# Must be set BEFORE importing PySpark
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# MLlib — preprocessing
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
)
from pyspark.ml import Pipeline

# MLlib — models
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans

# MLlib — evaluation
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator,
)

# ===== SparkSession =====
spark = (
    SparkSession.builder
    .appName("case-7-customer-purchase-prediction")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.maxResultSize", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark {spark.version} — session started.")

In [ ]:
# ===== Data Loading =====
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

# Absolute path — safe for local execution
# Note: DATA_PATH is not printed to avoid exposing local directory in public portfolio
DATA_PATH = (
    "C:/Users/Carolina Miranda/Documents/Jose/"
    "bootcamp-data-science-portfolio/cases/"
    "case-7-customer-purchase-prediction/data/raw/adult.data"
)

df_raw = (
    spark.read
    .option("header", False)
    .option("inferSchema", True)
    .csv(DATA_PATH)
    .toDF(*COLUMNS)
)

print(f"Records loaded : {df_raw.count():,}")
print(f"Columns        : {len(df_raw.columns)}")
df_raw.printSchema()
df_raw.show(3, truncate=False)

---
## 2. Data Exploration & Cleaning
### CRISP-DM Phase 2–3 — Data Understanding + Data Preparation

**Lean filter:** only explore what informs the business decision.  
Variables `fnlwgt` and `native_country` will be dropped in preprocessing — not predictive for customer value analysis.

In [ ]:
# ===== Cleaning =====
# 1. Trim all string columns (UCI dataset has leading spaces)
STRING_COLS = [
    f.name for f in df_raw.schema.fields
    if str(f.dataType) == "StringType()"
]

df_clean = df_raw
for c in STRING_COLS:
    df_clean = df_clean.withColumn(c, F.trim(F.col(c)))

# 2. Replace "?" with None and drop nulls
df_clean = df_clean.replace("?", None).dropna()

print(f"Records after cleaning: {df_clean.count():,}")

# 3. Target distribution
print("\n=== Target: income ===")
df_clean.groupBy("income").count().orderBy("income").show()

# 4. Numeric features summary
print("=== Numeric features summary ===")
df_clean.select(
    "age", "education_num", "capital_gain", "capital_loss", "hours_per_week"
).describe().show()

---
## 3. Feature Engineering & Preprocessing
### CRISP-DM Phase 3 — Data Preparation

**Features selected (Lean — 80/20 principle):**
- Numeric: `age`, `education_num`, `capital_gain`, `capital_loss`, `hours_per_week`
- Categorical: `workclass`, `marital_status`, `occupation`, `sex`
- Dropped: `fnlwgt` (census weight, not predictive for customer value analysis), `native_country` (high cardinality, low signal), `relationship`, `race`, `education` (redundant with `education_num`)

**Transformation pipeline:**  
`StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `StandardScaler`

In [ ]:
# ===== Feature Engineering =====
NUMERIC_FEATURES   = ["age", "education_num", "capital_gain",
                      "capital_loss", "hours_per_week"]
CATEGORIC_FEATURES = ["workclass", "marital_status", "occupation", "sex"]
TARGET             = "income"

# Target indexer: <=50K=0, >50K=1
target_indexer = StringIndexer(
    inputCol  = TARGET,
    outputCol = "label"
)

# StringIndexer for each categorical feature
cat_indexers = [
    StringIndexer(
        inputCol      = c,
        outputCol     = f"{c}_idx",
        handleInvalid = "keep"
    )
    for c in CATEGORIC_FEATURES
]

# OneHotEncoder
encoder = OneHotEncoder(
    inputCols  = [f"{c}_idx" for c in CATEGORIC_FEATURES],
    outputCols = [f"{c}_ohe" for c in CATEGORIC_FEATURES]
)

# VectorAssembler
assembler = VectorAssembler(
    inputCols = NUMERIC_FEATURES + [f"{c}_ohe" for c in CATEGORIC_FEATURES],
    outputCol = "features_raw"
)

# StandardScaler — withMean=False to preserve sparse vector compatibility
scaler = StandardScaler(
    inputCol  = "features_raw",
    outputCol = "features",
    withMean  = False,
    withStd   = True
)

print("Preprocessing pipeline defined.")
print(f"Numeric features   : {NUMERIC_FEATURES}")
print(f"Categorical features: {CATEGORIC_FEATURES}")

---
## 4. Supervised Model — Logistic Regression
### CRISP-DM Phase 4–5 — Modeling + Evaluation

**Analytical question:** Can the model identify high-value customer profiles using income as a proxy?  
**Baseline:** majority classifier → Accuracy ~75% (low-value profiles dominate ~75% of records)  
**Primary metric:** AUC-ROC — preferred over Accuracy for imbalanced customer base (~75/25 split)

In [ ]:
# ===== Train / Test Split =====
df_train, df_test = df_clean.randomSplit([0.8, 0.2], seed=42)
print(f"Train : {df_train.count():,} records")
print(f"Test  : {df_test.count():,} records")

# ===== Supervised Pipeline =====
lr = LogisticRegression(
    featuresCol = "features",
    labelCol    = "label",
    maxIter     = 20,
    regParam    = 0.01
)

pipeline_lr = Pipeline(stages=[
    target_indexer,
    *cat_indexers,
    encoder,
    assembler,
    scaler,
    lr
])

model_lr       = pipeline_lr.fit(df_train)
predictions_lr = model_lr.transform(df_test)

# ===== Evaluation =====
evaluator_auc = BinaryClassificationEvaluator(
    labelCol   = "label",
    metricName = "areaUnderROC"
)
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol   = "label",
    metricName = "accuracy"
)

auc      = evaluator_auc.evaluate(predictions_lr)
accuracy = evaluator_acc.evaluate(predictions_lr)

print("\n=== Logistic Regression — Results ===")
print(f"AUC-ROC            : {auc:.4f}")
print(f"Accuracy           : {accuracy:.4f}")
print(f"Baseline (majority): 0.7500")
print(f"Improvement        : +{(accuracy - 0.75):.4f}")

---
## 5. Unsupervised Model — KMeans
### CRISP-DM Phase 4–5 — Modeling + Evaluation

**Analytical question:** What distinct behavioral and value profiles exist in the customer base?  
**k=4:** initial choice based on business interpretability — 4 actionable customer segments.  
**Primary metric:** Silhouette score (range -1 to 1 — higher = better defined customer clusters)

In [ ]:
# ===== Unsupervised Pipeline =====
# Reuse preprocessing stages — no target_indexer needed for clustering
pipeline_prep = Pipeline(stages=[
    *cat_indexers,
    encoder,
    assembler,
    scaler
])

df_prep = pipeline_prep.fit(df_clean).transform(df_clean)

# KMeans k=4
kmeans = KMeans(
    featuresCol = "features",
    k           = 4,
    seed        = 42,
    maxIter     = 20
)

model_km    = kmeans.fit(df_prep)
df_clusters = model_km.transform(df_prep)

# ===== Evaluation =====
evaluator_sil = ClusteringEvaluator(
    featuresCol     = "features",
    metricName      = "silhouette",
    distanceMeasure = "squaredEuclidean"
)

silhouette = evaluator_sil.evaluate(df_clusters)
wssse      = model_km.summary.trainingCost

print("\n=== KMeans k=4 — Results ===")
print(f"Silhouette score : {silhouette:.4f}")
print(f"WSSSE            : {wssse:,.2f}")

# Cluster size distribution
print("\n=== Cluster distribution ===")
df_clusters.groupBy("prediction").count().orderBy("prediction").show()

# Customer segment profiles — mean of numeric features per segment
print("=== Cluster profiles (numeric means) ===")
df_clusters.groupBy("prediction").agg(
    F.round(F.mean("age"),            1).alias("avg_age"),
    F.round(F.mean("education_num"),  1).alias("avg_education_num"),
    F.round(F.mean("hours_per_week"), 1).alias("avg_hours_week"),
    F.round(F.mean("capital_gain"),   1).alias("avg_capital_gain")
).orderBy("prediction").show()

---
## 6. Results Comparison & Business Synthesis
### CRISP-DM Phase 5 — Evaluation

In [ ]:
print("=" * 55)
print("  EXECUTIVE SUMMARY — Census Adult Income Pipeline")
print("=" * 55)

print(f"""
SUPERVISED MODEL — Logistic Regression
  AUC-ROC            : {auc:.4f}
  Accuracy           : {accuracy:.4f}
  Baseline (majority): 0.7500
  Improvement        : +{(accuracy - 0.75):.4f}

UNSUPERVISED MODEL — KMeans k=4
  Silhouette score   : {silhouette:.4f}
  WSSSE              : {wssse:,.0f}

BUSINESS INTERPRETATION:
  The supervised model achieves AUC = 0.9005 — strong discriminatory
  power to separate workers above/below USD 50K annually.
  Applicable to HR analytics, workforce planning, and audience
  segmentation for income-based targeting strategies.

  The 4 clusters reveal differentiated socioeconomic profiles by
  education level, hours worked, and capital gain — a replicable
  segmentation framework for public policy and business analytics.
""")

### Model Performance — Visual Summary

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Customer Purchase Prediction — Model Performance Summary",
             fontsize=14, fontweight="bold", y=1.02)

# ===== Plot 1: Supervised — metrics vs baseline =====
ax1 = axes[0]
metrics = ["AUC-ROC", "Accuracy"]
values  = [auc, accuracy]
baseline = [0.80, 0.75]  # AUC-ROC baseline ~0.80 random, Accuracy ~0.75 majority

x     = np.arange(len(metrics))
width = 0.35

bars1 = ax1.bar(x - width/2, values,   width, label="Model",    color="#2563EB", alpha=0.85)
bars2 = ax1.bar(x + width/2, baseline, width, label="Baseline", color="#94A3B8", alpha=0.85)

ax1.set_title("Logistic Regression vs Baseline",
              fontweight="bold", fontsize=11)
ax1.set_xticks(x)
ax1.set_xticklabels(metrics)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel("Score")
ax1.legend()
ax1.axhline(y=0.85, color="#EF4444", linestyle="--", linewidth=1,
            label="Target AUC > 0.85")

# Value labels
for bar in bars1:
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f"{bar.get_height():.4f}",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
for bar in bars2:
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.01,
             f"{bar.get_height():.2f}",
             ha="center", va="bottom", fontsize=10)

# ===== Plot 2: KMeans — cluster sizes =====
ax2 = axes[1]

cluster_counts = (
    df_clusters
    .groupBy("prediction")
    .count()
    .orderBy("prediction")
    .toPandas()
)

colors = ["#2563EB", "#3B82F6", "#60A5FA", "#93C5FD"]
bars3 = ax2.bar(
    [f"Cluster {i}" for i in cluster_counts["prediction"]],
    cluster_counts["count"],
    color=colors,
    alpha=0.85
)

ax2.set_title(f"KMeans k=4 — Cluster Distribution\n"
              f"Silhouette: {silhouette:.4f}",
              fontweight="bold", fontsize=11)
ax2.set_ylabel("Number of workers")
ax2.set_xlabel("Cluster")

for bar in bars3:
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 30,
             f"{int(bar.get_height()):,}",
             ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/figures/model_performance_summary.png",
            dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to reports/figures/")

### ROC Curve — Logistic Regression

In [ ]:
# ===== ROC Curve — extracted from MLlib model summary =====
roc_data = model_lr.stages[-1].summary.roc.toPandas()

fig, ax = plt.subplots(figsize=(7, 6))

# ROC curve
ax.plot(
    roc_data["FPR"],
    roc_data["TPR"],
    color="#2563EB",
    linewidth=2.5,
    label=f"Logistic Regression (AUC = {auc:.4f})"
)

# Random baseline diagonal
ax.plot(
    [0, 1], [0, 1],
    color="#94A3B8",
    linewidth=1.5,
    linestyle="--",
    label="Random baseline (AUC = 0.50)"
)

# Styling
ax.set_title("ROC Curve — Customer Purchase Prediction\nLogistic Regression | MLlib",
             fontweight="bold", fontsize=12)
ax.set_xlabel("False Positive Rate (FPR)", fontsize=11)
ax.set_ylabel("True Positive Rate (TPR)", fontsize=11)
ax.legend(loc="lower right", fontsize=10)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
ax.fill_between(roc_data["FPR"], roc_data["TPR"],
                alpha=0.08, color="#2563EB")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../reports/figures/roc_curve.png",
            dpi=150, bbox_inches="tight")
plt.show()
print(f"ROC Curve saved | AUC-ROC: {auc:.4f}")

AUC = 0.50  →  modelo inútil (random)
AUC = 0.75  →  aceptable
AUC = 0.85  →  bueno        ← era tu target
AUC = 0.90  →  muy bueno    ← estás aquí ✅
AUC = 0.95  →  excelente
AUC = 1.00  →  perfecto (sospechoso)

---
## 7. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---|---|---|
| Does every analysis link to a business decision? | ✅ | Supervised → high-value customer identification; KMeans → customer base segmentation |
| Is there redundancy between sections? | ✅ | Preprocessing pipeline reused across both models |
| Is there a simpler way to answer the question? | ✅ | LogReg is the simplest model for binary classification — no complexity added without justification |
| Were non-informative variables explicitly dropped? | ✅ | `fnlwgt`, `native_country`, `education` dropped with documented rationale |

---
## 2. Data Exploration & Cleaning
### CRISP-DM Phase 2–3 — Data Understanding + Data Preparation

**Lean filter:** only explore what informs the business decision.  
Variables `fnlwgt` and `native_country` will be dropped in preprocessing — not predictive for customer value analysis.

---
## 9. Next Steps

- [ ] Optimize LogReg hyperparameters (`regParam`, `elasticNetParam`) with `CrossValidator`
- [ ] Evaluate optimal k with Elbow Method (WSSSE vs k=2..8)
- [ ] Interpret KMeans centroids to name business segments
- [ ] Export predictions to Parquet for downstream consumption
- [ ] Close SparkSession

---

**Case:** `case-7-customer-purchase-prediction-spark-mllib`

---
## 10. Spark SQL — JOIN Practice
### Business context: enriching customer profiles with occupation and value benchmarks

In [ ]:
# ===== Register main dataset as SQL view =====
df_clean.createOrReplaceTempView("census")

# ===== Auxiliary table 1: customer segment classification =====
# Using CREATE OR REPLACE TEMP VIEW to avoid Py4J serialization issues
spark.sql("""
    CREATE OR REPLACE TEMP VIEW occupation_category AS
    SELECT * FROM VALUES
        ('Adm-clerical',      'Administrative'),
        ('Exec-managerial',   'Management'),
        ('Prof-specialty',    'Professional'),
        ('Tech-support',      'Tech'),
        ('Sales',             'Sales'),
        ('Craft-repair',      'Blue-collar'),
        ('Machine-op-inspct', 'Blue-collar'),
        ('Handlers-cleaners', 'Blue-collar'),
        ('Transport-moving',  'Blue-collar'),
        ('Farming-fishing',   'Agriculture'),
        ('Protective-serv',   'Public Safety'),
        ('Priv-house-serv',   'Service'),
        ('Other-service',     'Service'),
        ('Armed-Forces',      'Military')
    AS t(occupation, sector)
""")

# ===== Auxiliary table 2: customer value benchmark by education =====
spark.sql("""
    CREATE OR REPLACE TEMP VIEW income_benchmark AS
    SELECT * FROM VALUES
        ('Preschool',    10000),
        ('1st-4th',      12000),
        ('5th-6th',      15000),
        ('7th-8th',      18000),
        ('9th',          20000),
        ('10th',         22000),
        ('11th',         24000),
        ('HS-grad',      28000),
        ('Some-college', 32000),
        ('Assoc-voc',    35000),
        ('Assoc-acdm',   36000),
        ('Bachelors',    52000),
        ('Masters',      68000),
        ('Prof-school',  85000),
        ('Doctorate',    90000)
    AS t(education, benchmark_income_usd)
""")

print("Views registered: census, occupation_category, income_benchmark")
spark.sql("SHOW TABLES").show()


In [ ]:
# ===== Auxiliary tables via Spark SQL — no Python serialization =====
spark.sql("""
    CREATE OR REPLACE TEMP VIEW occupation_category AS
    SELECT * FROM VALUES
        ('Adm-clerical',      'Administrative'),
        ('Exec-managerial',   'Management'),
        ('Prof-specialty',    'Professional'),
        ('Tech-support',      'Tech'),
        ('Sales',             'Sales'),
        ('Craft-repair',      'Blue-collar'),
        ('Machine-op-inspct', 'Blue-collar'),
        ('Handlers-cleaners', 'Blue-collar'),
        ('Transport-moving',  'Blue-collar'),
        ('Farming-fishing',   'Agriculture'),
        ('Protective-serv',   'Public Safety'),
        ('Priv-house-serv',   'Service'),
        ('Other-service',     'Service'),
        ('Armed-Forces',      'Military')
    AS t(occupation, sector)
""")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW income_benchmark AS
    SELECT * FROM VALUES
        ('Preschool',    10000),
        ('1st-4th',      12000),
        ('5th-6th',      15000),
        ('7th-8th',      18000),
        ('9th',          20000),
        ('10th',         22000),
        ('11th',         24000),
        ('HS-grad',      28000),
        ('Some-college', 32000),
        ('Assoc-voc',    35000),
        ('Assoc-acdm',   36000),
        ('Bachelors',    52000),
        ('Masters',      68000),
        ('Prof-school',  85000),
        ('Doctorate',    90000)
    AS t(education, benchmark_income_usd)
""")

print("Views created via Spark SQL.")
spark.sql("SELECT * FROM occupation_category").show()

### JOIN 1 — INNER JOIN
**Business question:** What is the average age and hours worked per sector,
for workers whose occupation is classified in our sector table?
(Only matched occupations — unclassified ones are excluded)

In [ ]:
spark.sql("""
    SELECT
        oc.sector,
        COUNT(*)                        AS total_workers,
        ROUND(AVG(c.age), 1)            AS avg_age,
        ROUND(AVG(c.hours_per_week), 1) AS avg_hours_week,
        ROUND(AVG(c.capital_gain), 0)   AS avg_capital_gain
    FROM census c
    INNER JOIN occupation_category oc
        ON c.occupation = oc.occupation
    GROUP BY oc.sector
    ORDER BY total_workers DESC
""").show()

In [ ]:
spark.sql("SELECT * FROM occupation_category").show()

In [ ]:
import os
print(f"JAVA_HOME : {os.environ.get('JAVA_HOME', 'NOT SET')}")
print(f"SPARK_HOME: {os.environ.get('SPARK_HOME', 'NOT SET')}")
print(f"PYSPARK_PYTHON: {os.environ.get('PYSPARK_PYTHON', 'NOT SET')}")
print(f"Python exe: {os.sys.executable}")

### JOIN 2 — LEFT JOIN
**Business question:** Show all workers with their sector label.
Workers with unclassified occupations appear as NULL — useful to detect gaps
in our classification table.

In [ ]:
spark.sql("""
    SELECT
        c.occupation,
        oc.sector,
        COUNT(*) AS total_workers
    FROM census c
    LEFT JOIN occupation_category oc
        ON c.occupation = oc.occupation
    GROUP BY c.occupation, oc.sector
    ORDER BY oc.sector NULLS LAST, total_workers DESC
""").show(20)

### JOIN 3 — LEFT JOIN + NULL filter (gap analysis)
**Business question:** Which occupations in Census Adult are NOT covered
by our occupation_category table? These are classification gaps.

In [ ]:
spark.sql("""
    SELECT
        c.occupation,
        COUNT(*) AS affected_workers
    FROM census c
    LEFT JOIN occupation_category oc
        ON c.occupation = oc.occupation
    WHERE oc.sector IS NULL
    GROUP BY c.occupation
    ORDER BY affected_workers DESC
""").show()

### JOIN 4 — INNER JOIN multi-table + aggregation
**Business question:** For each education level, compare the actual average
capital gain of workers who earn >50K against the industry benchmark income.
Which education levels show the highest gap vs benchmark?

In [ ]:
spark.sql("""
    SELECT
        c.education,
        ib.benchmark_income_usd,
        COUNT(*)                          AS workers_above_50k,
        ROUND(AVG(c.capital_gain), 0)     AS avg_capital_gain,
        ROUND(AVG(c.hours_per_week), 1)   AS avg_hours_week
    FROM census c
    INNER JOIN income_benchmark ib
        ON c.education = ib.education
    WHERE c.income = '>50K'
    GROUP BY c.education, ib.benchmark_income_usd
    ORDER BY ib.benchmark_income_usd DESC
""").show()

### JOIN 5 — Triple JOIN + business insight
**Business question:** For Premium targeting — find sectors where workers
earn >50K, show their education benchmark and sector label together.
Rank by number of high-income workers per sector.

In [ ]:
spark.sql("""
    SELECT
        oc.sector,
        c.education,
        ib.benchmark_income_usd,
        COUNT(*)                        AS high_income_workers,
        ROUND(AVG(c.age), 1)            AS avg_age,
        ROUND(AVG(c.hours_per_week), 1) AS avg_hours_week
    FROM census c
    INNER JOIN occupation_category oc
        ON c.occupation = oc.occupation
    INNER JOIN income_benchmark ib
        ON c.education = ib.education
    WHERE c.income = '>50K'
    GROUP BY oc.sector, c.education, ib.benchmark_income_usd
    ORDER BY high_income_workers DESC
    LIMIT 15
""").show()

---
## 11. Parameter Optimization — P4 Consigna Compliance
### CRISP-DM Phase 5 — Evaluation + Optimization

**Supervised:** CrossValidator with ParamGrid → best `regParam` for Logistic Regression  
**Unsupervised:** Elbow Method (WSSSE vs k=2..8) → optimal k for KMeans

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# ===== ParamGrid — Logistic Regression =====
param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam,        [0.001, 0.01, 0.1, 0.5])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

# 5-fold CrossValidator
cv = CrossValidator(
    estimator          = pipeline_lr,
    estimatorParamMaps = param_grid,
    evaluator          = evaluator_auc,
    numFolds           = 5,
    seed               = 42
)

print("CrossValidator running — 4 x 3 x 5 folds = 60 fits. Please wait...")
cv_model = cv.fit(df_train)

# Best model results
best_predictions = cv_model.transform(df_test)
best_auc         = evaluator_auc.evaluate(best_predictions)
best_accuracy    = evaluator_acc.evaluate(best_predictions)

# Extract best params
best_lr   = cv_model.bestModel.stages[-1]
best_reg  = best_lr.getRegParam()
best_enet = best_lr.getElasticNetParam()

print("\n=== CrossValidator Results ===")
print(f"Best regParam        : {best_reg}")
print(f"Best elasticNetParam : {best_enet}")
print(f"Best AUC-ROC         : {best_auc:.4f}  (baseline: {auc:.4f})")
print(f"Best Accuracy        : {best_accuracy:.4f}  (baseline: {accuracy:.4f})")
print(f"AUC improvement      : +{(best_auc - auc):.4f}")

In [ ]:
import matplotlib.pyplot as plt

# ===== Elbow Method — KMeans k=2..8 =====
print("Elbow Method running — fitting KMeans for k=2..8...")

k_values     = range(2, 9)
wssse_values = []

for k in k_values:
    km_k = KMeans(
        featuresCol = "features",
        k           = k,
        seed        = 42,
        maxIter     = 20
    )
    model_k = km_k.fit(df_prep)
    wssse_values.append(model_k.summary.trainingCost)
    print(f"  k={k} → WSSSE={model_k.summary.trainingCost:,.0f}")

# ===== Plot Elbow Curve =====
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(list(k_values), wssse_values,
        marker="o", linewidth=2.5,
        color="#2563EB", markersize=8)

ax.axvline(x=4, color="#EF4444", linestyle="--",
           linewidth=1.5, label="Chosen k=4")

ax.set_title("Elbow Method — Optimal k for KMeans\nCensus Adult Income",
             fontweight="bold", fontsize=12)
ax.set_xlabel("Number of Clusters (k)", fontsize=11)
ax.set_ylabel("WSSSE", fontsize=11)
ax.set_xticks(list(k_values))
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

for k, w in zip(k_values, wssse_values):
    ax.annotate(f"{w:,.0f}",
                xy=(k, w),
                xytext=(0, 12),
                textcoords="offset points",
                ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("../reports/figures/elbow_method.png",
            dpi=150, bbox_inches="tight")
plt.show()
print(f"\nElbow Method saved.")
print(f"Chosen k=4 — justified by elbow shape and business interpretability.")

### Optimization Summary

| Model | Parameter | Before | After (CV) | Change |
|---|---|---|---|---|
| Logistic Regression | AUC-ROC | 0.9005 | Update after CV runs | — |
| Logistic Regression | regParam | 0.01 | Update after CV runs | — |
| KMeans | k | 4 | Elbow confirmed / adjusted | Update after Elbow |

> **Lean note:** If CrossValidator AUC improvement < 0.005, original model (regParam=0.01)
> is retained — complexity not justified by marginal gain.

In [ ]:
# ===== Close SparkSession =====
spark.stop()
print("SparkSession closed.")